# Wildfire Incident Command — SFT Training

Supervised fine-tuning of **Qwen2.5-7B-Instruct** on wildfire incident command data.

- **Input:** `training/sft_data.jsonl` (generated by `scripts/generate_sft_data.py`)
- **Goal:** Teach the model to output valid JSON action objects given wildfire observations
- **Hardware:** A100 40GB on Colab

## Section 1 — Install Dependencies

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl==0.15.2 datasets==3.4.1

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not available — switch to a GPU runtime"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB")

## Section 2 — Load Model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

## Section 3 — Load Data

In [ ]:
import json
from datasets import Dataset
from collections import Counter

SFT_DATA_PATH = "training/sft_data.jsonl"

raw_examples = []
with open(SFT_DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        raw_examples.append(json.loads(line))

print(f"Loaded {len(raw_examples)} raw examples")

tier_dist = Counter(ex["tier"] for ex in raw_examples)
print(f"Tier distribution: {dict(tier_dist)}")

In [ ]:
def format_example(ex):
    """Format a single SFT example into a full conversation string for causal LM loss."""
    messages = ex["messages"]
    completion = ex["completion"]

    prompt_str = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = prompt_str + completion + tokenizer.eos_token
    return {"text": full_text}


formatted = [format_example(ex) for ex in raw_examples]
dataset = Dataset.from_list(formatted)

split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
val_dataset = split["test"]

print(f"Train: {len(train_dataset)}  |  Val: {len(val_dataset)}")
print(f"\nSample (first 500 chars):\n{formatted[0]['text'][:500]}")

## Section 4 — Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=True,
    args=TrainingArguments(
        output_dir="./sft_checkpoints",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        eval_strategy="steps",
        eval_steps=100,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        optim="adamw_8bit",
        seed=42,
    ),
)

print("Starting SFT training...")
trainer.train()
print("SFT training complete.")

## Section 5 — Quick Eval

Run 10 full episodes on easy tier with the trained model driving every step.
Requires env imports — upload the repo or clone it.

In [ ]:
import sys, os

# Adjust this path to wherever the repo root is in Colab
REPO_ROOT = "."  # or e.g. "/content/Wildfire-Containment-Simulator-main"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from env.wildfire_env import WildfireEnv
from env.serialization import serialize_observation
from env.action_parser import parse_action
from env.models import TIER_EASY

SYSTEM_PROMPT = (
    "You are an AI Incident Commander managing wildfire containment. "
    "You will receive a situation briefing each step. "
    "Respond with ONLY a valid JSON action object and nothing else. "
    'Example: {"action_type": "idle"}'
)

print("Env imports OK")

In [ ]:
import numpy as np

FastLanguageModel.for_inference(model)

EVAL_SEEDS = range(42, 52)
TIER = "easy"
MAX_STEPS = TIER_EASY.episode_length

rewards = []
pop_saved_pcts = []
parse_counts = {"json_success": 0, "regex_fallback": 0, "safe_idle": 0}
total_steps = 0

for seed in EVAL_SEEDS:
    env = WildfireEnv()
    obs = env.reset(task_id=TIER, seed=seed)
    episode_reward = 0.0
    step_num = 0
    prev_burning = 0

    while not env.done:
        prompt = serialize_observation(
            obs, step_num, MAX_STEPS,
            tier=TIER, prev_cells_burning=prev_burning,
        )
        prev_burning = obs.stats.cells_burning

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True,
            add_generation_prompt=True, return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            out = model.generate(
                input_ids,
                max_new_tokens=128,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

        action, status = parse_action(text, obs)
        parse_counts[status] = parse_counts.get(status, 0) + 1

        result = env.step(action)
        episode_reward += result.reward
        obs = result.observation
        step_num += 1

    total_steps += step_num
    rewards.append(episode_reward)

    state = env.state()
    total_pop = state["total_population"]
    pop_lost = state["population_lost"]
    pop_saved = 100.0 * (total_pop - pop_lost) / total_pop if total_pop > 0 else 100.0
    pop_saved_pcts.append(pop_saved)

    print(f"  Seed {seed}: reward={episode_reward:+.2f}, steps={step_num}, pop_saved={pop_saved:.0f}%")

mean_reward = np.mean(rewards)
std_reward = np.std(rewards)
total_parses = sum(parse_counts.values())
json_rate = 100.0 * parse_counts["json_success"] / total_parses if total_parses > 0 else 0
mean_pop = np.mean(pop_saved_pcts)

print(f"\n{'='*50}")
print(f"SFT Quick Eval — {TIER} tier, seeds {EVAL_SEEDS.start}-{EVAL_SEEDS.stop-1}")
print(f"Mean reward:       {mean_reward:+.2f} ± {std_reward:.2f}")
print(f"JSON success rate: {json_rate:.1f}%")
print(f"Mean pop saved:    {mean_pop:.1f}%")
print(f"Parse breakdown:   {dict(parse_counts)}")
print(f"{'='*50}")

assert mean_reward > 2.0, (
    f"SFT warm-up insufficient (mean_reward={mean_reward:.2f}) — do not proceed to GRPO"
)
print("\n✓ SFT checkpoint passes warm-up gate. Safe to proceed to GRPO.")

FastLanguageModel.for_training(model)

## Section 6 — Save & Export

In [ ]:
model.save_pretrained("./sft_final")
tokenizer.save_pretrained("./sft_final")
print("Saved to ./sft_final")

In [ ]:
# Push to HuggingFace Hub — replace with your username
HF_USERNAME = "Eshit"
model.push_to_hub(f"{HF_USERNAME}/wildfire-sft-7b")
tokenizer.push_to_hub(f"{HF_USERNAME}/wildfire-sft-7b")
print(f"Pushed to hub: {HF_USERNAME}/wildfire-sft-7b")

In [ ]:
!zip -r sft_final.zip ./sft_final
from google.colab import files
files.download("sft_final.zip")
print("Download started.")